# 04 — Kiểm định giả thiết mô hình (§4.2)

Trạng thái: **FROZEN** (2026-09-22, full run 150.000 khoản vay — xem `.claude/skills/notebook-first-model-dev/SKILL.md`).

Hai kiểm định bắt buộc: (1) tính thuần nhất theo thời gian (χ², Anderson-Goodman) chia estimation set theo năm; (2) bậc Markov 1 vs 2 (LR test). Dùng lại `chi2_homogeneity_test`/`lr_test_markov_order` từ Phase 0 (`scripts/utils/markov.py`).

In [1]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd())))

from utils.markov import chi2_homogeneity_test, lr_test_markov_order
from utils.transitions import (
    build_pair_counts,
    pairs_to_count_matrix,
    build_triple_counts,
    triples_to_count_matrix,
)

In [ ]:
SUFFIX = "full"  # FROZEN o full; doi lai "smoke" chi de kiem tra nhanh, khong ghi de artifact chinh thuc

DATA_DIR = os.path.join("..", "data", "processed")
TABLE_DIR = os.path.join("..", "outputs", "tables")
os.makedirs(TABLE_DIR, exist_ok=True)

STATE_NAMES = ["Current", "30 DPD", "60 DPD", "90+ DPD", "Default", "Prepaid"]
N_STATES = 6
ALPHA = 0.05
MERGE_THRESHOLD = 30  # nguong tong n_i. (4 hang tam thoi) de gop nam lien ke -- phan doan DRAFT, can HUNG review

## 1. Load estimation set

In [3]:
est_path = os.path.join(DATA_DIR, f"estimation_set_{SUFFIX}.parquet")
df = pd.read_parquet(est_path)
print(f"Đọc {est_path}: shape={df.shape}")
df["state"].value_counts().sort_index()

Đọc ..\data\processed\estimation_set_full.parquet: shape=(7253423, 6)


state
0    6915017
1      54919
2      16997
3      24977
4     136283
5     105230
Name: count, dtype: int64

## 2. Xây n_ij theo từng năm (giai đoạn con)

Gắn năm theo tháng $t+1$ (tháng xảy ra chuyển, đúng quy ước Ch.3 §3.2.2) — không lọc `df` theo năm trước khi ghép cặp, để không làm mất các cặp bắc cầu Tháng 12 → Tháng 1 năm sau.

In [4]:
pairs_1 = build_pair_counts(df, k=1)
pairs_1 = pairs_1.assign(year=((pairs_1["t1"] - 1) // 12))

year_to_nij = {}
for y, g in pairs_1.groupby("year"):
    year_to_nij[int(y)] = pairs_to_count_matrix(g, n_states=N_STATES)

summary_rows = []
for y in sorted(year_to_nij):
    n_transient = year_to_nij[y][:4].sum()
    summary_rows.append({"year": y, "n_transient_total": n_transient})
pd.DataFrame(summary_rows)

,year,n_transient_total
0,2016,215864.0
1,2017,779578.0
2,2018,1305158.0
3,2019,1554982.0
4,2020,1217775.0
5,2021,758303.0
6,2022,557535.0
7,2023,502540.0
8,2024,80133.0


## 3. Gộp giai đoạn ít quan sát (nếu cần)

**Phán đoán DRAFT (cần HUNG review):** gộp các năm liền kề nếu tổng quan sát 4 hàng tạm thời < `MERGE_THRESHOLD`. Đây là ngưỡng đơn giản để đảm bảo χ² không có ô kỳ vọng quá nhỏ, không phải quy tắc chính thức trong Ch.3 — nếu anh muốn ngưỡng khác, sửa `MERGE_THRESHOLD` ở cell constants.

In [5]:
years_sorted = sorted(year_to_nij)
periods = []  # list of (label, matrix)
buffer_mat = None
buffer_years = []
for y in years_sorted:
    mat = year_to_nij[y]
    buffer_mat = mat.copy() if buffer_mat is None else buffer_mat + mat
    buffer_years.append(y)
    if buffer_mat[:4].sum() >= MERGE_THRESHOLD:
        label = str(buffer_years[0]) if len(buffer_years) == 1 else f"{buffer_years[0]}-{buffer_years[-1]}"
        periods.append((label, buffer_mat))
        buffer_mat, buffer_years = None, []
if buffer_mat is not None:
    if periods:
        last_label, last_mat = periods[-1]
        new_label = f"{last_label.split('-')[0]}-{buffer_years[-1]}"
        periods[-1] = (new_label, last_mat + buffer_mat)
    else:
        label = str(buffer_years[0]) if len(buffer_years) == 1 else f"{buffer_years[0]}-{buffer_years[-1]}"
        periods.append((label, buffer_mat))

print(f"Số giai đoạn con sau khi gộp: {len(periods)}")
for label, mat in periods:
    print(f"  {label}: n_transient_total={mat[:4].sum():.0f}")

Số giai đoạn con sau khi gộp: 9
  2016: n_transient_total=215864
  2017: n_transient_total=779578
  2018: n_transient_total=1305158
  2019: n_transient_total=1554982
  2020: n_transient_total=1217775
  2021: n_transient_total=758303
  2022: n_transient_total=557535
  2023: n_transient_total=502540
  2024: n_transient_total=80133


## 4. Chạy χ² thuần nhất tổng thể

In [6]:
period_labels = [p[0] for p in periods]
period_matrices = [p[1] for p in periods]

stat, dof, p_value, reject = chi2_homogeneity_test(period_matrices, alpha=ALPHA)
conclusion = "BÁC BỎ H0 (không thuần nhất theo thời gian)" if reject else "KHÔNG BÁC BỎ H0 (thuần nhất theo thời gian)"
print(f"Giai đoạn: {period_labels}")
print(f"statistic={stat:.4f}, dof={dof}, p_value={p_value}, kết luận: {conclusion}")

mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)


Giai đoạn: ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
statistic=69101.6321, dof=160, p_value=0.0, kết luận: BÁC BỎ H0 (không thuần nhất theo thời gian)


## 5. χ² từng cặp giai đoạn liền kề (+ trước/sau 2020, Bonferroni)

In [7]:
pairwise_results = []
for i in range(len(periods) - 1):
    label_a, mat_a = periods[i]
    label_b, mat_b = periods[i + 1]
    s, d, p, r = chi2_homogeneity_test([mat_a, mat_b], alpha=ALPHA)
    pairwise_results.append({"pair": f"{label_a} vs {label_b}", "statistic": s, "dof": d, "p_value": p})

pre2020 = [mat for y, mat in year_to_nij.items() if y < 2020]
post2020 = [mat for y, mat in year_to_nij.items() if y >= 2020]
if pre2020 and post2020:
    mat_pre = sum(pre2020)
    mat_post = sum(post2020)
    s, d, p, r = chi2_homogeneity_test([mat_pre, mat_post], alpha=ALPHA)
    pairwise_results.append({"pair": "pre-2020 vs 2020+", "statistic": s, "dof": d, "p_value": p})

n_tests = len(pairwise_results)
alpha_bonf = ALPHA / n_tests if n_tests > 0 else ALPHA
pairwise_df = pd.DataFrame(pairwise_results)
if not pairwise_df.empty:
    pairwise_df["alpha_bonferroni"] = alpha_bonf
    pairwise_df["reject_H0"] = pairwise_df["p_value"] < alpha_bonf
pairwise_df.to_csv(os.path.join(TABLE_DIR, "chi2_homogeneity.csv"), index=False)
pairwise_df

mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)
mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)
mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)
mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)
mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)
mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)
mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)
mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)
mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)


,pair,statistic,dof,p_value,alpha_bonferroni,reject_H0
0,2016 vs 2017,115.655502,17,1.031940e-16,0.005556,True
1,2017 vs 2018,211.537659,19,1.704555e-34,0.005556,True
2,2018 vs 2019,4282.289911,19,0.000000e+00,0.005556,True
3,2019 vs 2020,19344.721842,20,0.000000e+00,0.005556,True
4,2020 vs 2021,2969.719389,20,0.000000e+00,0.005556,True
5,2021 vs 2022,6469.446353,20,0.000000e+00,0.005556,True
6,2022 vs 2023,1348.126450,20,1.453559e-273,0.005556,True
7,2023 vs 2024,26.255687,19,1.232208e-01,0.005556,False
8,pre-2020 vs 2020+,35627.601687,20,0.000000e+00,0.005556,True


## 6. Xây bộ ba n_ijk cho LR test bậc Markov

In [8]:
n_ij_overall = pairs_to_count_matrix(pairs_1, n_states=N_STATES)

triples = build_triple_counts(df)
n_ijk = triples_to_count_matrix(triples, n_states=N_STATES)
print(f"Số bộ ba hợp lệ: {len(triples)}")
print(f"Tổng n_ij (bậc 1, toàn estimation set): {n_ij_overall.sum():.0f}")

Số bộ ba hợp lệ: 6953525
Tổng n_ij (bậc 1, toàn estimation set): 7103422


## 7. Chạy LR test bậc Markov 1 vs 2

In [9]:
stat_o, dof_o, p_o, reject_o = lr_test_markov_order(n_ij_overall, n_ijk, alpha=ALPHA)
conclusion_o = "BÁC BỎ H0 (bậc 1 KHÔNG đủ, cần bậc 2)" if reject_o else "KHÔNG BÁC BỎ H0 (bậc 1 đủ)"
print(f"statistic={stat_o:.4f}, dof={dof_o}, p_value={p_o}, kết luận: {conclusion_o}")

order_test_summary = pd.DataFrame({
    "metric": ["statistic", "dof", "p_value", "reject_H0"],
    "value": [stat_o, dof_o, p_o, reject_o],
})
order_test_summary.to_csv(os.path.join(TABLE_DIR, "lr_test_order.csv"), index=False)
order_test_summary

mle_transition_matrix: trạng thái [5] không có quan sát chuyển nào (hàng NaN)


statistic=36899.9973, dof=54, p_value=0.0, kết luận: BÁC BỎ H0 (bậc 1 KHÔNG đủ, cần bậc 2)


,metric,value
0,statistic,36899.997299
1,dof,54
2,p_value,0.0
3,reject_H0,True


## 8. Quyết định

Quyết định: Chấp nhận kết quả 2 kiểm định này làm kết luận chính thức cho §4.2. Cả hai đều **bác bỏ H0** — không tự động coi đây là lý do phải sửa mô hình (build Markov bậc 2/semi-Markov nằm ngoài phạm vi dự án, xem `PROJECT_BRIEF.md` mục 6) — mà ghi nhận trung thực vào Ch.4 (kết quả + diễn giải) và Ch.5 (hạn chế + hướng phát triển chỉ nêu, không triển khai).

Lý do:
- **χ² thuần nhất theo thời gian:** statistic=69.101,63, dof=160, p≈0 → bác bỏ H0 tổng thể. Xét từng cặp năm liền kề (Bonferroni, α=0,05/9=0,005556): **8/9 cặp bác bỏ**, chỉ riêng 2023-2024 không bác bỏ (p=0,123). Khác với bản smoke (chỉ 2019-2022 bác bỏ), ở full run hầu hết các cặp đều bác bỏ — đây là dấu hiệu kinh điển của **cỡ mẫu lớn làm p-value rất nhạy**: với hàng triệu quan sát, ngay cả khác biệt nhỏ về mặt thực chất cũng đủ để bác bỏ H0 về mặt thống kê. Không nên diễn giải "8/9 cặp năm khác nhau hoàn toàn" theo nghĩa thực chất — nên nhấn mạnh **độ lớn khác biệt** (statistic của 2019-2020 = 19.344, của 2023-2024 chỉ 26) hơn là chỉ nhìn accept/reject nhị phân. Giai đoạn 2018-2022 (bao trùm COVID) có statistic lớn vượt trội so với các giai đoạn khác — nhất quán với giả thuyết forbearance/thiên tai.
- **LR test bậc Markov 1 vs 2:** statistic=36.899,997, dof=54, p≈0 → bác bỏ H0, bậc 1 không đủ. Khớp đúng tín hiệu đã thấy ở Phase 2 (Chapman-Kolmogorov: Frobenius norm 0,42 không giảm theo N).
- **Kết luận chung cho báo cáo:** mô hình Markov bậc 1 thuần nhất theo thời gian là một **giả định đơn giản hóa không hoàn toàn đúng** với dữ liệu — cả 2 kiểm định đều bác bỏ H0 với p-value cực nhỏ, nhất quán với nhau và với tín hiệu CK ở Phase 2. Đây là hạn chế cần nêu rõ trong Ch.5, không phải lý do dừng dự án — đúng tinh thần "kiểm tra giả thiết mô hình" mà đề bài yêu cầu.
- Ngưỡng `MERGE_THRESHOLD=30` không có tác dụng ở full run (mọi năm đều vượt xa ngưỡng, ít nhất 80.133 quan sát ở 2024) nên không cần điều chỉnh.